# Overfitting & Validation : Notebook de la matinée
## Épisodes 0 à 4

**L'histoire en quatre lignes.** Un élève reçoit un livre d'exercices corrigés. Chaque vendredi, contrôle. Avant qu'il ouvre le livre, le prof-magicien a fait disparaître 5–6 exercices par vendredi : l'élève ne les verra jamais. À la fin de l'année, examen sur un *autre* livre, mêmes concepts.

**Le magicien a deux pouvoirs, et deux seulement.**
1. **Le tour** : faire disparaître les exercices du vendredi. `magicien()`. Trois lignes ; vous l'appelez vous-mêmes.
2. **Le mode omniscient** : connaître d'avance les concepts de tous les livres possibles. `verite_du_magicien()` : la note face à la vérité absolue du magicien ; à ne pas confondre avec l'examen final, vrai livre fini que l'élève passera. Ligne pointillée sur les graphiques. **Jamais** dans une cellule où l'on choisit quelque chose.

**Le bruit.** Les corrigés imprimés contiennent des fautes de frappe et d'impression : le corrigé dit 9,81 là où les concepts donnent 9,79. C'est le **bruit** (amplitude σ = 0.3). Il ne dépend ni de l'élève ni de sa façon d'apprendre, et le vendredi, on est noté *contre le corrigé imprimé*.

**Les deux échelles.** La machine mesure une **erreur quadratique** : 0 = parfait, pas de plafond, plus bas = mieux. Le prof convertit en **note sur 20** avec un barème fixe (`en_note`) : 0/20 = l'élève qui récite la même réponse partout ; moins d'erreur = meilleure note. Dans ce notebook, **les tables parlent en /20, les graphiques en erreur** (l'axe des graphiques est donc « renversé » : le bas, c'est bien). Une seule fois, à l'épisode 2, les deux vues seront superposées pour apprendre à passer de l'une à l'autre.

**Comment lire ce notebook.** Chaque épisode enchaîne les mêmes cellules :

| Étiquette | Ce qu'on fait |
|---|---|
| **Exécutée** | Le code est complet. On l'exécute, on lit le graphique. |
| **À trous** | Compléter les `___`. Ce qui manque est *le principe de l'épisode*, pas la plomberie. |
| **Ce que vous devez voir** | L'observation attendue. Vérifiez-la vous-même avant de continuer. |
| **Auto-explication** | Deux lignes, dans les mots de l'histoire. |
| **[L3+] Extension** | Sans squelette. |
| **[M2] Plafond** | Tâche ouverte. |

**Le fil de la matinée.**

| Épisode | Question de l'histoire | Ce qu'on apprend |
|---|---|---|
| **0** | Le décor : livre, tour du magicien, examen scellé | Le protocole entraînement / validation / test |
| **1** | Trois élèves lisent le même livre : qui réussit vendredi ? | Sous-apprentissage, sur-apprentissage, généralisation |
| **2** | Après un mauvais vendredi, que faire ? | Les deux leviers : plus d'exercices, ou moins de règles retenues |
| **3** | Et si toute la classe révisait pareil ? | L'erreur se décompose : biais² + variance + bruit |
| **4** | Pourquoi le tour du magicien est-il fait ainsi ? | **La validation croisée** : pourquoi retirer avant, pourquoi plusieurs vendredis |

---
## Épisode 0. Prologue : le livre, le tour, le sceau

**Exécutée** : tout le décor tient dans cette cellule. Lisez `magicien` (trois lignes) et `verite_du_magicien` (le mode omniscient) avant d'exécuter.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

SIGMA = 0.3          # amplitude du bruit : les fautes de frappe dans les corrigés
N_LIVRE = 30         # exercices par livre
N_VENDREDIS = 5      # 5 vendredis x 6 exercices


def concepts(x):
    """Les concepts de physique. Réservé au mode omniscient du magicien."""
    return np.sin(2 * np.pi * x)


def fonds_exercices(n, seed):
    """Tire n exercices du fonds du prof : énoncé x, corrigé y (avec son bruit)."""
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 1, size=n)
    y = concepts(x) + rng.normal(0, SIGMA, size=n)
    return x[:, None], y


def magicien(X, y, vendredi):
    """Pouvoir 1, le tour : retire les exercices du vendredi demandé (1..N_VENDREDIS).
    Trois lignes. N'importe quel prof peut le faire."""
    plis = list(KFold(N_VENDREDIS, shuffle=True, random_state=0).split(X))
    idx_visible, idx_disparus = plis[vendredi - 1]
    return (X[idx_visible], y[idx_visible]), (X[idx_disparus], y[idx_disparus])


_X_INFINI, _Y_INFINI = fonds_exercices(100_000, seed=424242)   # « une infinité de livres »


def verite_du_magicien(eleve):
    """Pouvoir 2, le mode omniscient : la note de l'élève face à la vérité absolue
    du magicien (10^5 exercices frais). À ne pas confondre avec l'examen final,
    qui est un livre fini que l'élève passera vraiment.
    Cellules d'affichage uniquement. Ne sert JAMAIS à choisir quoi que ce soit."""
    return mean_squared_error(_Y_INFINI, eleve.predict(_X_INFINI))


BAREME = 0.5 + SIGMA**2   # variance des corrigés du fonds : réciter la réponse moyenne partout -> 0/20


def en_note(erreur):
    """Le barème du prof : convertit l'erreur de la machine en note sur 20.
    20/20 = zéro erreur ; 0/20 = réciter la même réponse à tous les exercices.
    Toujours : moins d'erreur = meilleure note. Le barème ne change jamais.
    Accepte un nombre ou un tableau."""
    return np.maximum(0.0, 20 * (1 - np.asarray(erreur) / BAREME))


sceau_leve = False


def examen_final():
    """L'autre livre, mêmes concepts. Scellé jusqu'à l'épisode 5."""
    if not sceau_leve:
        raise RuntimeError("L'examen final est scellé jusqu'à l'épisode 5.")
    return fonds_exercices(N_LIVRE, seed=2025)


def eleve(nb_regles):
    """Un élève défini par le nombre de règles qu'il retient de ses révisions :
    1 règle = l'élève naïf ; quelques-unes = le généraliste ; beaucoup = le par-cœur.
    (Sous le capot : un polynôme, la révélation est en fin de notebook.
    Recentrage des énoncés sur [-1, 1] puis mise à l'échelle des puissances : sans effet
    sur la courbe, seulement sur la stabilité du calcul : le vrai par-cœur, sur toutes les machines.)"""
    return make_pipeline(MinMaxScaler(feature_range=(-1, 1)),
                         PolynomialFeatures(nb_regles, include_bias=False),
                         StandardScaler(),
                         LinearRegression())


X_livre, y_livre = fonds_exercices(N_LIVRE, seed=16)                      # le livre est imprimé...
(X_vis, y_vis), (X_disp, y_disp) = magicien(X_livre, y_livre, vendredi=1)  # ...le tour est fait AVANT
print(f"Livre : {len(X_livre)} exercices. Visibles : {len(X_vis)}. Disparus pour le vendredi 1 : {len(X_disp)}.")

ModuleNotFoundError: No module named 'numpy'

**Les deux échelles.** La machine mesure une **erreur quadratique** : 0 = parfait, pas de plafond, plus bas = mieux. Le prof convertit en **note sur 20** avec un barème fixe (`en_note`) : 0/20 = l'élève qui récite la même réponse partout ; moins d'erreur = meilleure note. Dans ce notebook, **les tables parlent en /20, les graphiques en erreur** (l'axe des graphiques est donc « renversé » : le bas, c'est bien). Une seule fois, à l'épisode 2, les deux vues seront superposées pour apprendre à passer de l'une à l'autre.

**Exécutée** : le livre tel que l'élève le voit. Les six exercices disparus ne sont pas affichés : l'élève ne les verra jamais.

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(X_vis, y_vis, color="k", label=f"exercices visibles ({len(X_vis)})")
plt.xlabel("énoncé x"); plt.ylabel("corrigé y")
plt.title("Le livre, tel que l'élève le voit (vendredi 1)")
plt.legend(); plt.show()

**Exécutée** : l'examen final existe déjà, mais il est scellé. On ne l'ouvrira qu'à l'épisode 5, une seule fois.

In [ ]:
try:
    examen_final()
except RuntimeError as e:
    print("→", e)

---
## Épisode 1 : Comprendre ou retenir

Trois élèves ouvrent le même livre et n'en retiennent pas la même chose. **L'élève naïf** résume tout le livre en une seule règle. **Le généraliste** en retient trois, prudentes et générales. **L'élève par cœur** retient une règle par exercice ou presque : quinze, fautes de frappe comprises. Plus on retient de règles, plus on peut coller au livre ; reste à voir le vendredi.

**Exécutée** : les trois élèves après entraînement sur les 24 exercices visibles. En pointillé bleu, ce que seul le magicien sait : les concepts. Dans le titre, la note face à la vérité du magicien.

In [ ]:
profils = {"naïf": 1, "généraliste": 3, "par cœur": 15}    # nombre de règles retenues par chacun
pluriel = lambda r: f"{r} règle" + ("s" if r > 1 else "")
grille = np.linspace(0, 1, 300)[:, None]

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, (nom, r) in zip(axes, profils.items()):
    el = eleve(r).fit(X_vis, y_vis)                       # l'élève ne voit que le visible
    ax.scatter(X_vis, y_vis, color="k", s=18, label="livre (visible)")
    ax.plot(grille, el.predict(grille), color="C3", label=f"élève {nom}")
    ax.plot(grille, concepts(grille), "--", color="C0", label="concepts (mode omniscient)")
    ax.set_ylim(-2, 2); ax.set_xlabel("x")
    ax.set_title(f"{nom} ({pluriel(r)})\nvérité du magicien : {en_note(verite_du_magicien(el)):.1f}/20", fontsize=10)
axes[0].set_ylabel("y"); axes[0].legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

**À trous** : la table des notes. La note sur le livre est calculée sur les exercices visibles ; complétez la note du vendredi, calculée sur les exercices **disparus**.

In [ ]:
note_et_erreur = lambda err: f"{en_note(err):4.1f}/20 (err {err:7.3f})"
print(f"{'élève':>22} | {'note livre':>22} | {'note vendredi':>22} | {'vérité du magicien':>22}")
for nom, r in profils.items():
    el = eleve(r).fit(X_vis, y_vis)
    err_livre    = mean_squared_error(y_vis, el.predict(X_vis))
    err_vendredi = mean_squared_error(___, el.predict(___))     # À COMPLÉTER : les exercices disparus
    print(f"{nom + ' (' + pluriel(r) + ')':>22} | {note_et_erreur(err_livre):>22} | {note_et_erreur(err_vendredi):>22} | {note_et_erreur(verite_du_magicien(el)):>22}")

**Ce que vous devez voir.** Plus un élève retient de règles, meilleure est sa note sur le livre : le par-cœur frôle 19/20. Le vendredi, c'est le généraliste qui gagne, et le par-cœur tombe à **0/20**. La vérité du magicien raconte la même histoire que le vendredi, pas que le livre.

Vocabulaire, posé une fois : **sous-apprentissage** = mauvais partout (le naïf) ; **sur-apprentissage** = excellent sur le livre, mauvais le vendredi (le par-cœur) ; **généralisation** = la note du vendredi, la seule qui compte.

**Auto-explication** : *Pourquoi l'élève par cœur a-t-il la meilleure note sur le livre et la pire le vendredi ?* Deux lignes, dans les mots de l'histoire.

> Votre réponse :

**[L3+] Extension** : le même par-cœur, un livre plus gros. Le prof sort un livre de 100 exercices, puis de 300, puis de 1000, même tour du magicien, mêmes 15 règles. Complétez la table (note livre, note vendredi, vérité, avec l'erreur entre parenthèses, comme la table ci-dessus) et répondez : **à partir de quand le par-cœur devient-il acceptable ? Est-ce encore du par-cœur ?**

**Ce que vous devez voir.** Sur 24 exercices, deux zéros qui n'ont rien à voir : vendredi 0/20 (erreur ≈ 5) et vérité 0/20 (erreur ≈ 17,5), le barème est borné à 0, c'est l'erreur qui les distingue. Puis, à mesure que le livre grossit, le vendredi du par-cœur remonte jusqu'à ≈ 17/20 et sa vérité colle au plafond du bruit, comme le généraliste. Il est devenu acceptable, au prix de trente fois plus d'exercices que ce que trois règles demandaient. Retenez la formule pour l'épisode 2 : *« par cœur » n'est pas une propriété de l'élève, c'est un rapport entre le nombre de règles retenues et le nombre d'exercices travaillés.*

In [ ]:
# Votre code ici


---
## Épisode 2 : Deux façons de progresser

Après un mauvais vendredi, l'élève a deux leviers, et deux seulement : demander au prof **le tome 2** (plus d'exercices), ou **s'interdire de retenir les détails** (retenir moins de règles). Chaque levier a sa courbe.

Dans cet épisode, la note du vendredi est la **moyenne des 5 vendredis** : le même tour que `magicien`, sur les 5 découpages possibles (`KFold(5)`). Attention, ce ne sont pas cinq semaines qui se suivent : ce sont cinq élèves indépendants, un par découpage, qui ne se transmettent rien. Pourquoi c'est mieux qu'un seul vendredi, et pourquoi ils doivent rester indépendants : épisode 4.

**Exécutée** : levier 2, le nombre de règles retenues, de 1 à 15. Pour chacun : erreur sur le livre, moyenne des 5 vendredis, et vérité du magicien (moyenne des 5 élèves entraînés).

In [ ]:
from sklearn.model_selection import validation_curve

nb_regles_grille = np.arange(1, 16)
vendredis = KFold(N_VENDREDIS, shuffle=True, random_state=0)      # le même tour que magicien()

train_scores, val_scores = validation_curve(
    eleve(1), X_livre, y_livre,
    param_name="polynomialfeatures__degree", param_range=nb_regles_grille,   # degree = nb de règles (cf. la révélation)
    cv=vendredis, scoring="neg_mean_squared_error")
err_livre     = -train_scores.mean(axis=1)
err_vendredis = -val_scores.mean(axis=1)

# Mode omniscient (affichage seulement) : les mêmes élèves face à la vérité du magicien
erreur_verite = np.array([
    np.mean([verite_du_magicien(eleve(r).fit(X_livre[tr], y_livre[tr])) for tr, _ in vendredis.split(X_livre)])
    for r in nb_regles_grille])

fig, (ax_err, ax_note) = plt.subplots(2, 1, figsize=(7.5, 7.5), sharex=True, gridspec_kw={"hspace": 0.15})

# En haut : la vue de la machine (celle que vous verrez partout dans le métier)
ax_err.plot(nb_regles_grille, err_livre, "o-", label="erreur sur le livre")
ax_err.plot(nb_regles_grille, err_vendredis, "s-", label="erreur du vendredi (moyenne des 5)")
ax_err.plot(nb_regles_grille, erreur_verite, "--", color="k", label="vérité du magicien")
ax_err.axhline(SIGMA**2, color="gray", ls=":", label="bruit σ²")
ax_err.set_yscale("log"); ax_err.set_ylabel("erreur quadratique\n(plus bas = mieux)")
ax_err.set_title("Levier 2 : la capacité, courbe de validation"); ax_err.legend(fontsize=8)

# En bas : exactement les mêmes courbes, passées au barème du prof
ax_note.plot(nb_regles_grille, en_note(err_livre), "o-", label="note sur le livre")
ax_note.plot(nb_regles_grille, en_note(err_vendredis), "s-", label="note du vendredi (moyenne des 5)")
ax_note.plot(nb_regles_grille, en_note(erreur_verite), "--", color="k", label="vérité du magicien")
ax_note.axhline(float(en_note(SIGMA**2)), color="gray", ls=":", label="plafond ≈ 17/20 (bruit)")
ax_note.set_ylim(-0.5, 20); ax_note.set_ylabel("note /20\n(plus haut = mieux)")
ax_note.set_xlabel("nombre de règles retenues (capacité de l'élève)")
ax_note.set_title("La même courbe, au barème du prof", fontsize=10); ax_note.legend(fontsize=8)
plt.show()
r_star = nb_regles_grille[np.argmin(err_vendredis)]
print(f"Nombre de règles préféré par les vendredis : {r_star}, note du vendredi {en_note(err_vendredis.min()):.1f}/20, vérité du magicien {en_note(erreur_verite[np.argmin(err_vendredis)]):.1f}/20")

**Ce que vous devez voir.** L'erreur sur le livre descend à chaque règle supplémentaire, sans jamais remonter. L'erreur du vendredi descend, touche son minimum vers 3 règles, puis remonte, doucement d'abord, violemment au-delà de 8, quand le nombre de règles se rapproche du nombre d'exercices du livre : le par-cœur commence là. La vérité du magicien fait la même chose, un peu au-dessus : le vendredi est une bonne estimation, pas la vérité. Personne ne passe durablement sous la ligne σ², au barème, plancher d'erreur = **plafond de note ≈ 17/20** face à la vérité du magicien. Pourquoi le 20/20 est impossible : épisode 3, le bruit.

**Lecture du panneau du bas.** La note culmine chez le généraliste puis s'effondre. Mais regardez au-delà de 8 règles : tous les élèves affichent 0/20 alors que leurs erreurs, en haut, diffèrent d'un facteur 100. Le barème **sature** ; l'erreur, non. C'est exactement pour ça que le métier trace l'erreur et pas la note : à partir de l'épisode 3, seuls les graphiques en erreur restent, et le barème continue de vivre dans les tables.

**À trous** : levier 1, le nombre d'exercices. Le prof sort les tomes suivants (250 exercices). Le nombre d'exercices travaillés (`nb_exercices`, de 20 à 200) est l'axe de la courbe. Complétez le nombre de règles des deux élèves : le généraliste, et un par-cœur (9 règles suffisent ici pour garder le graphique lisible).

In [ ]:
from sklearn.model_selection import learning_curve

X_tomes, y_tomes = fonds_exercices(250, seed=2)           # les tomes suivants
rng = np.random.default_rng(0)
sous_livres = [rng.permutation(len(X_tomes)) for _ in range(N_VENDREDIS)]   # 5 façons de choisir n exercices (mode omniscient)

nb_exercices = np.array([20, 32, 48, 64, 96, 128, 160, 200])   # les tailles de livre qu'on va essayer

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, (nom, r) in zip(axes, {"généraliste": ___, "par cœur": ___}.items()):   # À COMPLÉTER : combien de règles chacun ?
    n_ex, tr, va = learning_curve(
        eleve(r), X_tomes, y_tomes,
        train_sizes=nb_exercices,                          # l'axe de la courbe : le nombre d'exercices travaillés
        cv=vendredis, scoring="neg_mean_squared_error", shuffle=True, random_state=0)
    verite = [np.mean([verite_du_magicien(eleve(r).fit(X_tomes[p[:n]], y_tomes[p[:n]])) for p in sous_livres]) for n in n_ex]
    ax.plot(n_ex, -tr.mean(axis=1), "o-", label="erreur sur le livre")
    ax.plot(n_ex, -va.mean(axis=1), "s-", label="erreur du vendredi (moyenne des 5)")
    ax.plot(n_ex, verite, "--", color="k", label="vérité du magicien")
    ax.axhline(SIGMA**2, color="gray", ls=":", label="bruit σ²")
    ax.set_yscale("log"); ax.set_xlabel("nombre d'exercices travaillés (n)"); ax.set_title(f"élève {nom} ({r} règles)")
axes[0].set_ylabel("erreur quadratique (plus bas = meilleure note)"); axes[0].legend(fontsize=8)
plt.suptitle("Levier 1 : le nombre d'exercices, courbe d'apprentissage"); plt.tight_layout(); plt.show()

**Ce que vous devez voir.** Chez le généraliste, les trois courbes se rejoignent près de la ligne σ² dès quelques dizaines d'exercices. Chez le par-cœur, l'écart entre le livre et le vendredi est énorme à petit n et se referme lentement, mais il se referme. Le levier *n* agit sur l'écart ; le levier *capacité* agit sur l'endroit où se trouve le minimum de la courbe précédente.

L'axe horizontal ici est **le nombre d'exercices**. Vous reverrez le mot « courbe d'apprentissage » plus tard dans la formation avec un autre axe : ce ne sera pas la même courbe.

**Auto-explication** : *L'élève par cœur reçoit le tome 2. Son écart livre/vendredi augmente ou diminue ? Pourquoi, dans les mots de l'histoire ?*

> Votre réponse :

**[L3+] Extension** : tracez directement l'écart *vérité du magicien − erreur sur le livre* (c'est R(f̂) − R̂ₙ(f̂), l'encadré), d'abord en fonction du nombre de règles, avec les tableaux de la courbe de validation ; puis en fonction de n pour l'élève à 9 règles. Vérifiez qu'il croît avec la capacité et décroît avec n.

In [ ]:
# Votre code ici


---
## Épisode 3 : La classe entière

Jusqu'ici, un seul élève. Maintenant, **cinquante élèves**, chacun avec son propre livre tiré du même fonds. Tous révisent de la même façon : même nombre de règles retenues. Le magicien les compare sur un exercice qu'aucun n'a vu.

- Les cinquante **naïfs** répondent presque la même chose… et se trompent **tous pareil**. Ce n'est pas le livre qui les pénalise, c'est leur façon de réviser. On appelle ça le **biais**.
- Les cinquante **par-cœur** donnent cinquante réponses différentes, éparpillées. Chacun a mémorisé *son* livre. On appelle ça la **variance**.
- Et par-dessus tout ça, il reste le **bruit** : on est noté contre le corrigé imprimé, fautes de frappe comprises.

**Exécutée** : 50 livres, 50 élèves par profil. En rouge pâle les 50 réponses, en rouge épais leur moyenne, en pointillé bleu les concepts.

In [ ]:
N_CLASSE = 50
grille = np.linspace(0, 1, 200)[:, None]
verite_grille = concepts(grille).ravel()
profils = {"naïf": 1, "généraliste": 3, "par cœur": 9}

reponses = {}          # profil -> tableau (50 élèves x 200 points)
for nom, r in profils.items():
    reponses[nom] = np.array([
        eleve(r).fit(*fonds_exercices(N_LIVRE, seed=1000 + i)).predict(grille).ravel()
        for i in range(N_CLASSE)])

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, (nom, r) in zip(axes, profils.items()):
    for reponse in reponses[nom]:
        ax.plot(grille, reponse, color="C3", alpha=0.15, lw=1)
    ax.plot(grille, reponses[nom].mean(axis=0), color="C3", lw=2.5, label="réponse moyenne des 50")
    ax.plot(grille, verite_grille, "--", color="C0", lw=2, label="concepts (vérité)")
    ax.set_ylim(-2.2, 2.2); ax.set_xlabel("x")
    ax.set_title(f"{nom} ({r} règle{'s' if r > 1 else ''})")
axes[0].set_ylabel("y"); axes[0].legend(fontsize=8)
plt.suptitle("50 élèves, 50 livres, la même façon de réviser"); plt.tight_layout(); plt.show()

**À trous** : mettez des chiffres sur ce que vous venez de voir, pour chaque profil.

- Le **biais²** mesure de combien la *réponse moyenne de la classe* rate les concepts. C'est un écart entre `reponses[nom].mean(axis=0)` et `verite_grille`, au carré, moyenné sur la grille.
- La **variance** mesure de combien les élèves diffèrent *entre eux*. C'est la dispersion des 50 réponses en chaque point de la grille, moyennée sur la grille.

In [ ]:
print(f"{'profil':>22} | {'biais²':>9} | {'variance':>9} | {'bruit σ²':>9} | {'somme':>9} | {'vérité mesurée':>15}")
for nom, r in profils.items():
    biais2   = np.mean(___)          # À COMPLÉTER : la réponse moyenne rate-t-elle les concepts ?
    variance = np.mean(___)          # À COMPLÉTER : les 50 élèves diffèrent-ils entre eux ?
    somme = biais2 + variance + SIGMA**2
    mesuree = np.mean([verite_du_magicien(eleve(r).fit(*fonds_exercices(N_LIVRE, seed=1000 + i)))
                       for i in range(N_CLASSE)])
    print(f"{nom + ' (' + str(r) + ')':>22} | {biais2:9.3f} | {variance:9.3f} | {SIGMA**2:9.3f} | {somme:9.3f} | {mesuree:15.3f}")

**Ce que vous devez voir.** Le naïf a un gros **biais²** (0.20) et une petite variance (0.02) : les 50 droites sont serrées les unes contre les autres, mais aucune ne suit les concepts. Le par-cœur a un biais² 2 fois plus petit et une **variance** 150 fois plus grosse : la moyenne des 50 suit à peu près la courbe bleue, mais chaque élève pris individuellement part dans sa direction, surtout aux bords du livre. Le généraliste a les deux petits. Et surtout : **biais² + variance + bruit ≈ la vérité mesurée**. Ce n'est pas une coïncidence, c'est une identité : c'est *toute* l'erreur, décomposée en trois parts.

**Exécutée** : les trois parts en fonction du nombre de règles. La courbe « somme » est celle que vous avez déjà vue en U à l'épisode 2 : on vient de l'ouvrir.

In [ ]:
N_LIVRES_MC = 200                       # plus de livres = décomposition plus stable
regles_grille = np.arange(1, 11)

biais2s, variances = [], []
for r in regles_grille:
    P = np.array([eleve(r).fit(*fonds_exercices(N_LIVRE, seed=2000 + i)).predict(grille).ravel()
                  for i in range(N_LIVRES_MC)])
    biais2s.append(np.mean((P.mean(axis=0) - verite_grille) ** 2))
    variances.append(np.mean(P.var(axis=0)))
biais2s, variances = np.array(biais2s), np.array(variances)

plt.figure(figsize=(7.5, 5))
plt.plot(regles_grille, biais2s, "o-", label="biais² (la méthode est-elle bonne ?)")
plt.plot(regles_grille, variances, "s-", label="variance (le livre reçu décide-t-il ?)")
plt.axhline(SIGMA**2, color="gray", ls=":", label="bruit σ² (rien à y faire)")
plt.plot(regles_grille, biais2s + variances + SIGMA**2, "--", color="k", label="somme = erreur totale")
plt.yscale("log"); plt.xlabel("nombre de règles retenues"); plt.ylabel("erreur quadratique (plus bas = mieux)")
plt.title("Les trois parts de l'erreur"); plt.legend(fontsize=8); plt.show()

r_opt = regles_grille[np.argmin(biais2s + variances)]
print(f"Meilleur compromis : {r_opt} règles.")

**Ce que vous devez voir.** Le biais² s'effondre dès 3 règles et reste au plancher (attention à l'échelle : elle est logarithmique, ces variations-là ne pèsent rien). La variance, elle, ne fait que monter, et explose au-delà de 7 règles. La courbe « somme » se confond alors avec la variance : passé ce point, toute l'erreur vient du livre reçu. Le minimum de la somme est un **compromis** : il n'existe pas de réglage qui annule les deux. Et la ligne du bruit ne bouge jamais.

**Où l'histoire casse (1/2).** Un élève peut toujours réviser plus, mieux, autrement : il agit sur le biais et sur la variance. Le bruit, lui, n'est la faute de personne et ne bouge pas d'un iota. L'histoire de l'élève n'a pas de place pour une erreur qui n'appartient à personne ; le modèle, si : c'est σ². C'est pour ça que le 20/20 est impossible (plafond ≈ 17/20 au barème).

**Auto-explication** : *Le prof donne le tome 2 à toute la classe, quelle part de l'erreur baisse ? Il interdit le par-cœur (moins de règles) : quelle part baisse, quelle part monte ?*

> Votre réponse :

**[L3+] Encadré : d'où sort l'identité.** Notons *L* le livre reçu (aléatoire), f̂_L l'élève entraîné dessus, et y = f(x) + ε avec ε de moyenne nulle et de variance σ², indépendant de *L*. En ajoutant et retranchant la réponse moyenne E_L[f̂_L(x)] :

E[(y − f̂_L(x))²] = (f(x) − E_L[f̂_L(x)])² + Var_L[f̂_L(x)] + σ²

Les termes croisés s'annulent : ε est indépendant de *L* et centré, et f̂_L − E_L[f̂_L] est centré par construction. Les trois morceaux sont, dans l'ordre : **biais²**, **variance**, **bruit** : les trois colonnes de votre table. *On revient à l'élève.*

**[M2] Plafond** : dans la vraie vie, on n'a **qu'un seul livre**. La tentation est de simuler les 50 élèves en tirant 50 fois *avec remise* dans ce livre unique (bootstrap) au lieu de tirer 50 livres frais. Implémentez-le sur `X_livre, y_livre` et comparez à la décomposition ci-dessus pour 1, 3, 6 et 9 règles. Où l'écart devient-il grave, et pourquoi ?

In [ ]:
# Votre code ici


---
## Épisode 4. Pourquoi le magicien : la validation croisée

> **Si vous ne deviez retenir qu'une section de la matinée, c'est celle-ci.** Les épisodes 1 à 3 expliquaient *ce qui se passe* dans la tête de l'élève (par cœur, biais, variance). L'épisode 4 explique *comment on mesure* : c'est le protocole que vous appliquerez sur chaque projet, et il porte un nom : la **validation croisée** (*cross-validation*).

**Le problème.** L'élève veut savoir s'il a compris. Il ne peut pas se noter sur le livre : il connaît les corrigés, il aurait 19/20 en récitant (épisode 1). Il ne peut pas non plus utiliser l'examen final : il n'a lieu qu'une fois, à la fin. Il lui faut donc une note *maintenant*, à partir du seul livre dont il dispose. C'est exactement ce que le tour du magicien fabrique.

**Le tour, en une phrase.** Le magicien coupe le livre en k paquets. Il crée alors **k mondes parallèles**. Dans le monde n° 1, l'élève révise sur les paquets 2 à k et passe son vendredi sur le paquet 1. Dans le monde n° 2, un **autre** élève, même méthode, mémoire vierge, révise sur les paquets 1, 3, 4… et passe son vendredi sur le paquet 2. Et ainsi de suite. La note finale est la moyenne des k notes.

> ⚠️ **Il n'y a pas de chronologie.** C'est le contresens le plus fréquent, et le récit du prologue y invite : on imagine un élève qui passe le vendredi 1, lit sa note, s'améliore, passe le vendredi 2… **Ce n'est pas ce qui se produit.** Les k élèves ne se connaissent pas, ne se transmettent rien, et aucun ne voit la note d'un autre. Si vous préférez une seule ligne du temps : le magicien **efface la mémoire** de l'élève après chaque vendredi, avant qu'il ne recommence à réviser sur un autre découpage. Les deux images disent la même chose : l'important est qu'**aucune information ne circule d'un vendredi à l'autre**.
>
> C'est ce qui rend la moyenne honnête. Si l'élève s'améliorait d'un vendredi au suivant, les k notes ne mesureraient pas la même chose et leur moyenne ne voudrait rien dire. (L'élève qui *apprend de ses notes*, lui, existe bel et bien : c'est l'épisode 5, et c'est un tout autre problème.)

**Quatre questions, dans l'ordre.**

**(a) Pourquoi retirer les exercices *avant* que l'élève révise ?** Parce que s'il les avait vus, le vendredi mesurerait sa mémoire et non sa compréhension. C'est la règle d'or : ce qui sert à évaluer ne doit pas avoir servi à apprendre. Dans chacun des k mondes, le découpage est fait avant l'ajustement : c'est ce que garantissent `KFold` et, plus tard, le `Pipeline`.

**(b) Pourquoi k mondes, et pas un seul ?** Six exercices, c'est peu. L'élève qui tombe sur six exercices faciles se croit meilleur qu'il n'est ; celui qui tombe sur six pièges se croit nul. La note existe, mais elle est **tirée au sort** : et vous allez mesurer de combien elle bouge.

**(c) La rotation.** Chaque exercice du livre sert de contrôle dans exactement un monde, et de révision dans les k−1 autres. On n'a pas plus d'exercices : on les utilise mieux. C'est le **k** de « validation croisée à k plis » : ici k = 5.

**(d) Jusqu'où pousser ?** Un seul exercice en contrôle par monde, trente mondes : la note la plus fidèle, au prix de trente ajustements. Ça porte un nom aussi (*leave-one-out*), et vous l'écrirez dans la cellule à trous.

**Le vocabulaire, posé ici une fois pour toutes** : un paquet mis de côté = un **pli** (*fold*), et c'est le **jeu de validation** de son monde ; les k mondes = la **validation croisée à k plis** ; la note obtenue = une **estimation** de la vraie erreur, avec sa propre dispersion.

**Exécutée** : la preuve que les mondes sont bien parallèles, `cross_validate` peut rendre les k élèves qu'il a fabriqués. Regardez leurs règles.

In [ ]:
from sklearn.model_selection import cross_validate

resultat = cross_validate(eleve(3), X_livre, y_livre,
                          cv=KFold(5, shuffle=True, random_state=0),
                          scoring="neg_mean_squared_error",
                          return_estimator=True)      # rend les k élèves fabriqués

print(f"Nombre d'élèves fabriqués par une validation croisée à 5 plis : {len(resultat['estimator'])}\n")
print(f"{'monde':>6} | {'les 3 règles retenues par CET élève':>40} | {'sa note du vendredi':>19}")
for i, (eleve_du_monde, score) in enumerate(zip(resultat["estimator"], resultat["test_score"]), start=1):
    regles = ", ".join(f"{c:+6.2f}" for c in eleve_du_monde[-1].coef_)
    print(f"{i:6d} | {regles:>40} | {en_note(-score):17.1f}/20")

print("\nCinq élèves distincts, entraînés chacun dans son monde. Aucun n'a vu la note des autres.")
print(f"Note retenue = moyenne des cinq : {en_note(-resultat['test_score'].mean()):.1f}/20")

**Mais au fait, pourquoi se donner tout ce mal ?** La question ne vient pas de l'élève : un élève n'a pas l'initiative de sa configuration, il révise comme il révise. Elle vient du **prof**. C'est lui qui a devant lui dix configurations d'élèves (1 à 10 règles retenues) et qui doit décider laquelle mérite le livre qu'il a fait imprimer. *Retenez la distribution des rôles, elle est exactement celle du métier : l'élève est le modèle, il n'a aucune initiative ; le prof est le data scientist, c'est lui qui compare des configurations et qui retient la meilleure.*

**Le scénario.** Premier vendredi de l'année. Sur sa copie du classement, un profil se démarque : le prudent, à 3 règles. Le prof pourrait s'arrêter là. Mais il se méfie : une seule mesure, six exercices tirés au sort… est-ce le **profil** qui est bon, ou le **tirage** qui lui a souri ? Les vérités qu'il connaît en mode omniscient donnent raison à sa prudence : les configurations 3, 4 et 5 valent 0.105, 0.107, 0.107 ; deux millièmes d'écart. Un classement établi sur une mesure qui se balade de 14 à 19/20 désignera un vainqueur, mais au hasard.

**L'expérience.** Pour tester la **stabilité** de son vainqueur, le prof rejoue le tour **300 fois sur le même livre** : 300 fois, il refait le choix complet parmi les dix configurations, et note qui gagne. D'abord avec le protocole paresseux (un seul vendredi), puis avec le tour complet (5 mondes). Si « 3 règles » est une propriété du livre et non du tirage, ce profil doit sortir presque à chaque fois.

In [ ]:
from sklearn.model_selection import ShuffleSplit, cross_val_score

candidats = np.arange(1, 11)

# Ce que le magicien sait : le meilleur candidat, entraîné sur tout le livre
verites = np.array([verite_du_magicien(eleve(r).fit(X_livre, y_livre)) for r in candidats])
meilleur = candidats[np.argmin(verites)]
print(f"Vérité du magicien, meilleure configuration : {meilleur} règles.")
print("  candidats 3, 4, 5 :", " ".join(f"{v:.3f}" for v in verites[2:5]), "← séparés par des millièmes\n")

protocoles_choix = {
    "1 vendredi (6 exercices)": lambda s: ShuffleSplit(n_splits=1, test_size=6, random_state=s),
    "5 mondes (KFold à 5 plis)": lambda s: KFold(5, shuffle=True, random_state=s),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (nom, faire_le_tour) in zip(axes, protocoles_choix.items()):
    gagnants = []
    for s in range(300):                       # 300 fois : et si le magicien avait coupé autrement ?
        notes = [-cross_val_score(eleve(r), X_livre, y_livre, cv=faire_le_tour(s),
                                  scoring="neg_mean_squared_error").mean() for r in candidats]
        gagnants.append(candidats[int(np.argmin(notes))])
    ax.hist(gagnants, bins=np.arange(0.5, 11.5), color="C0", edgecolor="white")
    ax.axvline(meilleur, color="k", ls="--", label=f"bon choix = {meilleur}")
    ax.set_xlabel("configuration désignée (nombre de règles)"); ax.set_title(nom)
    taux = 100 * np.mean(np.array(gagnants) == meilleur)
    ax.text(0.98, 0.92, f"bon choix : {taux:.0f} %", transform=ax.transAxes, ha="right", fontweight="bold")
axes[0].set_ylabel("nombre de fois sur 300"); axes[0].legend(fontsize=8)
plt.suptitle("Le même livre, la même décision, prise 300 fois par le prof"); plt.tight_layout(); plt.show()

**Ce que vous devez voir.** Avec un seul vendredi, le vainqueur change sans arrêt : le bon profil ne sort que **43 fois sur 100**, et le classement couronne 69 fois une configuration à 6 règles ou plus, des par-cœur désignés sur un coup de chance. Avec cinq mondes, le bon profil sort **84 fois sur 100** ; sur les 49 erreurs restantes, 46 désignent le voisin immédiat (4 règles), dont la vérité est à deux millièmes de l'optimum, et 3 seulement vont plus loin. Le vainqueur du premier vendredi est confirmé, mais c'est le protocole à k mondes qui pouvait le confirmer, pas le vendredi qui l'avait désigné.

**Trois précisions qui comptent :**
1. **La stabilité est une évidence, pas une preuve.** Que « 3 règles » sorte 8 fois sur 10 montre que le protocole ne désigne pas au hasard. C'est la vérité du magicien, un luxe des données synthétiques, qui certifie que c'est le bon. Dans la vraie vie, vous n'aurez que la stabilité.
2. **On ne rejoue pas 300 fois en vrai.** Cette expérience est celle qui *justifie l'habitude* : une validation croisée à 5 ou 10 plis, une fois, en sachant ce qu'elle achète. La dispersion des k notes par pli vous donne déjà un aperçu de la marge d'erreur.
3. **Comparer est plus exigeant que noter.** Une comparaison a besoin d'une mesure plus fine que l'écart entre candidats ; et quand deux candidats sont à égalité dans la marge d'erreur, le classement ne tranche pas : c'est au prof de trancher, sur un autre critère (le plus simple, le plus rapide, le plus interprétable).

C'est la distinction classique de la littérature : la validation croisée sert soit à **estimer le risque** d'un modèle, soit à **sélectionner** parmi plusieurs (Arlot & Celisse, *Statistics Surveys* 2010) ; deux usages légitimes, dont la sélection est le dominant dans la pratique quotidienne. Et notez qui s'est souvenu des notes tout au long de cette section : le prof. Un prof qui compare, choisit, recommence… consomme les vendredis. Ce que ça lui coûtera, c'est l'épisode 5.

**Exécutée** : maintenant la question (b), de combien la note bouge-t-elle ? Un seul élève, un seul livre, le profil généraliste. On rejoue **100 fois le tour du magicien**, 100 jeux de k mondes, en ne changeant que le découpage.

In [ ]:
from sklearn.model_selection import ShuffleSplit, RepeatedKFold, LeaveOneOut, cross_val_score

N_REPLICATIONS = 100
eleve_teste = eleve(3)                                     # le généraliste
verite = verite_du_magicien(eleve(3).fit(X_livre, y_livre))  # ce que le magicien sait de CET élève

protocoles = {
    "1 vendredi\n(6 exercices)": lambda s: ShuffleSplit(n_splits=1, test_size=6, random_state=s),
    "5 vendredis":               lambda s: KFold(5,  shuffle=True, random_state=s),
    "10 vendredis":              lambda s: KFold(10, shuffle=True, random_state=s),
    "5 vendredis\nx 10 rotations": lambda s: RepeatedKFold(n_splits=5, n_repeats=10, random_state=s),
}

notes = {}
for nom, faire_le_tour in protocoles.items():
    notes[nom] = [-cross_val_score(eleve_teste, X_livre, y_livre, cv=faire_le_tour(s),
                                   scoring="neg_mean_squared_error").mean()
                  for s in range(N_REPLICATIONS)]           # random_state DIFFÉRENT à chaque réplication

plt.figure(figsize=(8, 4.5))
plt.boxplot(notes.values(), tick_labels=notes.keys())
plt.axhline(verite, color="k", ls="--", label="vérité du magicien pour cet élève")
plt.ylabel("erreur estimée (plus bas = meilleure note)")
plt.title(f"La même mesure, refaite {N_REPLICATIONS} fois"); plt.legend(fontsize=8); plt.show()

for nom, v in notes.items():
    v = np.array(v)
    print(f"{nom.replace(chr(10), ' '):28s} : de {en_note(v.max()):4.1f}/20 à {en_note(v.min()):4.1f}/20")

**À trous** : le protocole (d), un exercice par vendredi, trente vendredis. C'est un `KFold` poussé à son maximum, et scikit-learn lui donne un nom. Complétez le découpage, puis le nombre de contrôles que ça représente.

In [ ]:
erreur_loo = -cross_val_score(eleve_teste, X_livre, y_livre,
                              cv=___,                       # À COMPLÉTER : un exercice retiré à la fois
                              scoring="neg_mean_squared_error").mean()
nb_controles = ___                                          # À COMPLÉTER : combien de contrôles cela fait-il ?
print(f"30 vendredis : {erreur_loo:.3f} soit {en_note(erreur_loo):.1f}/20, en {nb_controles} contrôles.")
print(f"Vérité du magicien : {verite:.3f} soit {en_note(verite):.1f}/20. Aucune réplication : ce protocole est déterministe.")

**Ce que vous devez voir.** Les quatre boîtes sont **centrées au même endroit** : les protocoles mesurent tous la même chose. Ce qui change, c'est leur **hauteur**. Avec un seul vendredi, la note va de 14/20 à 19/20 selon les six exercices tirés : le même élève, le même livre, cinq points d'écart. Avec cinq vendredis, l'écart tombe à moins de deux points ; avec dix rotations, à un demi-point. Le protocole (d) ne donne qu'une valeur : il n'y a rien à tirer au sort quand chaque exercice sert une fois.

Notez aussi que la médiane ne tombe pas pile sur la vérité du magicien. C'est normal : chaque protocole entraîne l'élève sur *moins* de 30 exercices, et sur ce livre-ci le hasard fait le reste. Une note reste une **estimation** : à reporter avec sa dispersion, pas toute seule.

**Auto-explication** : *Deux élèves passent chacun un seul vendredi. L'un a 15/20, l'autre 12/20. Peut-on dire lequel est le meilleur ?*

> Votre réponse :

**[L3+] Encadré, comment choisit-on k ?** Deux forces opposées. Avec *k* petit, chaque élève est entraîné sur beaucoup moins d'exercices que le livre entier (à k=2, sur la moitié) : on mesure un élève affaibli, donc l'estimation est **pessimiste**. Avec *k* grand, l'entraînement ressemble au livre entier, le biais s'efface, mais il faut *k* ajustements, et les *k* élèves entraînés se ressemblent tellement que leurs erreurs sont fortement corrélées. En pratique : **k = 5 ou 10**, et si le budget le permet, on **répète** les rotations pour réduire ce qui reste de hasard dans le découpage.

**[L3+] Extension** : mesurez le pessimisme, pour k = 2, 3, 5, 10 puis 30, tirez 60 livres neufs et comparez la moyenne des notes estimées à la vérité du magicien du même élève. Le pessimisme décroît-il comme annoncé ?

In [ ]:
# Votre code ici


**[M2] Plafond** : le graphique précédent fixe le livre et ne fait varier que le tour ; l'extension ci-dessus fait l'inverse. Superposez les deux sources : pour k = 5, décomposez la variance totale de la note estimée en « part due au livre » et « part due au découpage » (une petite analyse de variance à deux facteurs suffit : 30 livres × 20 découpages). Laquelle domine ici ? Que faudrait-il changer au protocole pour que l'autre devienne dominante ?

---
## La révélation : « règles » = degré

L'élève à *r* règles est, sous le capot, un **polynôme de degré r** : réponse(x) ≈ a₁·x + a₂·x² + … + a_r·x^r. Chaque règle retenue = un coefficient ajustable de plus. Le mot du métier est **degré**, et la notion générale, combien de règles un modèle *peut* retenir, s'appelle la **capacité** (ou complexité) du modèle. À partir de l'épisode 3, les deux vocabulaires cohabitent.

---
## Lexique FR ↔ EN (toute la matinée)

| Dans l'histoire | En français technique | En anglais (doc, API) |
|---|---|---|
| Le livre visible | Jeu d'entraînement | Training set |
| Les exercices disparus | Jeu de validation | Validation set |
| L'autre livre, scellé | Jeu de test | Test set |
| Les fautes de frappe des corrigés | Bruit irréductible, σ² | Noise, irreducible error |
| La vérité du magicien | La vraie erreur de généralisation | True generalization error |
| Ne garder qu'une règle pour tout | Sous-apprentissage | Underfitting |
| Apprendre les corrigés par cœur | Sur-apprentissage | Overfitting |
| La note du vendredi (la seule qui compte) | Généralisation | Generalization |
| Le nombre de règles retenues | Degré du polynôme ; capacité (complexité) du modèle | Polynomial degree; model capacity |
| Tous les élèves se trompent pareil | Biais | Bias |
| Chacun a mémorisé son livre | Variance | Variance |
| Le compromis à trouver | Compromis biais-variance | Bias–variance tradeoff |
| Un seul vendredi, tiré au sort | Découpage simple | Holdout, `ShuffleSplit` |
| La rotation des vendredis | Validation croisée à k plis | k-fold cross-validation, `KFold` |
| Un vendredi de la rotation | Un pli | A fold |
| Refaire la rotation plusieurs fois | Validation croisée répétée | `RepeatedKFold` |
| Un exercice par vendredi | Validation croisée « un contre tous » | Leave-one-out, `LeaveOneOut` |
| Rejouer le tour du magicien | Réplication, graine aléatoire | Replication, `random_state` |
| Erreur au carré, moyennée | Erreur quadratique moyenne | MSE, `neg_mean_squared_error` |
| Courbe « erreur vs capacité » | Courbe de validation | `validation_curve` |
| Courbe « erreur vs nombre d'exercices » | Courbe d'apprentissage | `learning_curve` (`train_sizes`) |
| La chaîne élève complète | Chaîne de traitement | `Pipeline`, `make_pipeline` |

---
*Fin de la matinée. L'après-midi : épisode 5, le prof qui se règle sur les vendredis, et la levée du sceau ; puis les fuites, la fiche d'une page, et le défi diagnostic.*